36. 有效的数独

https://leetcode.cn/problems/valid-sudoku/description/?envType=study-plan-v2&envId=top-interview-150

请你判断一个 9 x 9 的数独是否有效。只需要根据规则，验证已经填入的数字是否有效即可。

In [1]:
from typing import List

class Solution:
    def isValidSudoku(self, board: List[List[str]]) -> bool:
        rows = [set() for _ in range(9)]
        cols = [set() for _ in range(9)]
        boxes = [set() for _ in range(9)]

        for i in range(9):
            for j in range(9):
                num = board[i][j]
                if num == ".":
                    continue
                box_index = (i // 3) * 3 + j // 3
                if num in rows[i] or num in cols[j] or num in boxes[box_index]:
                    return False
                rows[i].add(num)
                cols[j].add(num)
                boxes[box_index].add(num)
        return True

In [7]:
[x for x in range((5 + 1) // 2)]

[0, 1, 2]

54. 螺旋矩阵

https://leetcode.cn/problems/spiral-matrix/description/?envType=study-plan-v2&envId=top-interview-150

如下是我的写法。问题不明显，但是运行了边界条件才发现，如果矩阵是列数很少、行数很多，有时候最后会出现只剩一列的情况，我们的代码没有单独判断。且，判断条件也应该是 `(min(m, n) + 1) // 2` 作为边界而不是简单的只是行数，但整体思路（所谓剥洋葱思想）没问题。

In [8]:
from typing import List

class Solution:
    def spiralOrder(self, matrix: List[List[int]]) -> List[int]:
        answer = []
        i = 0
        while i < (len(matrix) + 1) // 2:
            dual = len(matrix) - 1 - i
            if dual == i:
                for j in range(i, len(matrix[0]) - i):
                    answer.append(matrix[i][j])
                break
            else:
                for j in range(i, len(matrix[0]) - i):
                    answer.append(matrix[i][j])
                for _ in range(i + 1, len(matrix) - i):
                    answer.append(matrix[_][len(matrix[0]) - i - 1])
                for j in range(len(matrix[0]) - i - 2, i - 1, -1):
                    answer.append(matrix[dual][j])
                for _ in range(dual - 1, i, -1):
                    answer.append(matrix[_][i])
            i += 1
        return answer
    
solution = Solution()
matrix = [[1,2,3,4],[5,6,7,8],[9,10,11,12]]
print(solution.spiralOrder(matrix))

[1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]


需要改进的是，对于只剩一行、只剩一列、其他三种情况的顺序处理，以及对于指针边界的约束，其他的没问题；如下。

In [9]:
class Solution:
    def spiralOrder(self, matrix: List[List[int]]) -> List[int]:
        answer = []
        i = 0
        m, n = len(matrix), len(matrix[0])
        while i < (min(m, n) + 1) // 2:
            top, bottom = i, m - i - 1
            left, right = i, n - i - 1

            if top == bottom:
                for j in range(left, right + 1):
                    answer.append(matrix[top][j])
                break

            if left == right:
                for r in range(top, bottom + 1):
                    answer.append(matrix[r][left])
                break
            
            for j in range(left, right + 1):
                answer.append(matrix[top][j])
            
            for r in range(top + 1, bottom + 1):
                answer.append(matrix[r][right])

            for j in range(right - 1, left - 1, -1):
                answer.append(matrix[bottom][j])

            for r in range(bottom - 1, top, -1):
                answer.append(matrix[r][left])

            i += 1
        
        return answer

48. 旋转图像

https://leetcode.cn/problems/rotate-image/description/?envType=study-plan-v2&envId=top-interview-150

本题引发了一个数学问题的思考。在此之前，我们先写一遍标准做法。实际上，这道题在之前旋转字符串时见到过，旋转就等于转置 + 水平翻转，这是核心思路。如下。

In [2]:
from typing import List

class Solution:
    def rotate(self, matrix: List[List[int]]) -> None:
        """
        Do not return anything, modify matrix in-place instead.
        """
        for i in range(1, len(matrix)):
            for j in range(i):
                matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]

        for i in range(len(matrix)):
            matrix[i].reverse()

然而，究竟为什么非得是转置 + 翻转，而不是其他操作呢？

实际上，我们知道旋转的单射，即每个点的坐标的去向；不过这样我们就需要 n * n 个临时变量来作为 swap 动作的存储空间，也就是 n * n 大小的临时矩阵了；注意，此时不存在所谓的 async 并发交换，例如

In [ ]:
import asyncio

matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

async def f(i, j): # some rotation function
    return j, len(matrix) - 1 - i

async def move(i,j):
    ni, nj = f(i,j)
    matrix[ni][nj] = matrix[i][j]

tasks = [move(i, j) for i in range(len(matrix)) for j in range(len(matrix[0]))]
await asyncio.gather(*tasks)

这实际上是行不通的，理论上会存在所谓数据竞争（race condition），且 python 的 asyncio 只是协作式调度，不是 CPU 并行的。

这里我们可以发现，一个操作如果涉及到两两交换、三三交换甚至更多，那么这一组交换中我们就只需要使用一个 temp 来作为存储；如果一个矩阵的旋转操作可以被解耦成若干次这样的操作，每次操作都只需要一个临时 temp，那么最终空间复杂度仍旧是 o(1) 的，只是重复了几次循环，时间复杂度上升。

这种能够解耦的操作，实际上就是置换（permutation），在数学上称作 cycle decomposition of permutation，一旦置换环（cycle）出现就可以解决问题。而刚刚所看到的置换实际上只是置换群 $S_{n^2}$ 中的一个元素。

让我们正式介绍 Cycle Leader Algorithm（循环领导者算法），它可以用 O(1) 的额外空间对任意 permutation 进行 in-place 重排。刚刚的旋转算法只是其中的一种。

问题的数学形式如下。假设我们有一个数组 $A = [a_0, a_1, ..., a_{n-1}]$，我们知道一个**置换函数**
$$
f : {0..n-1} \rightarrow {0..n-1}
$$
目标是得到 $A'[f(i)] = A[i]$。为了方便，如下我们考虑 A 是一维数据，即数组。

此时，如果直接 `A[f(i)] = A[i]` 会发生 overwrite 的问题，所以需要利用一核心数学定理，即任何 permutation 都可以分解为**若干个 cycle**。例如
```
0 -> 2 -> 3 -> 1 -> 0
```
这是一个 **cycle**
```
(0 2 3 1)
```
数学上
$$
S_n = cycle_1 \cup cycle_2 \cup ...
$$

如下给出 CLA 的伪代码。假设我们有 permutation `f(i)`。

In [5]:
def apply_permutation(arr, f):
    n = len(arr)
    visited = [False]*n
    for start in range(n):
        if visited[start]:
            continue
        current = start
        temp = arr[start]
        while True:
            next_pos = f(current)
            if next_pos == start:
                arr[current] = temp
                break
            arr[current] = arr[next_pos]
            visited[current] = True
            current = next_pos
        visited[current] = True

上面用了 `visited`，但其实可以**不用 visited**，标记已经访问的元素，例如 `arr[i] = -arr[i]`，这样就能在数组内部记录状态。

Cycle Leader Algorithm 在很多地方出现
- 原地洗牌，Fisher-Yates 的某些变种；
- FFT bit-reversal permutation，快速傅里叶变换；
- 数据库和内存重排，如 `column-store -> row-store`；
- GPU memory layout transform，例如 `NHWC <-> NCHW`。